# 04. Scheduling and the Sweep Line

Scheduling problems choose which jobs to run. The **sweep line** technique turns intervals into **events** (start = +1, end = -1) and scans them in time order.

**Learning order:** ACTIVITY SELECTION → SWEEP LINE → PLATFORMS → CAR FLEET → MY CALENDAR → INTERVIEW THINKING

## 1. Activity Selection

Choose the maximum number of non-overlapping events. Sort by **finish time**, pick an event if it starts after the last chosen one ends. This is the classical greedy with a clean exchange proof: replacing the first event of any optimal solution with the earliest-finishing event never causes a clash.

In [1]:
def max_events(events):
    events.sort(key=lambda e: e[1])
    count, last_end = 0, float("-inf")
    for start, end in events:
        if start >= last_end:
            count += 1
            last_end = end
    return count

print(max_events([(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]))   # 4

4


## 2. Sweep line: maximum overlap

Turn each interval into two events: `(start, +1)` and `(end, -1)`. Sort events by time. Scan and keep a running count. The **maximum running count** is the maximum number of overlapping intervals (= meeting rooms needed).

With half-open intervals `[start, end)`, at the same time an **end must come before a start**. Sorting `(time, delta)` puts `-1` before `+1` automatically.

In [2]:
def max_overlap(intervals):
    events = []
    for s, e in intervals:
        events.append((s, 1))
        events.append((e, -1))
    events.sort()
    cur = best = 0
    for _, delta in events:
        cur += delta
        best = max(best, cur)
    return best

print(max_overlap([[0, 30], [5, 10], [15, 20]]), max_overlap([[1, 2], [2, 3]]))   # 2 1

2 1


## 3. Minimum Platforms (arrivals and departures)

Same idea with two **sorted** arrays: arrivals and departures. If the next arrival is `<=` the next departure, a platform is needed (`+1`); otherwise a train leaves (`-1`). Here touching times count as overlapping (a train arriving at the same time another leaves needs a second platform).

In [3]:
def min_platforms(arr, dep):
    arr, dep = sorted(arr), sorted(dep)
    i = j = plat = best = 0
    while i < len(arr):
        if arr[i] <= dep[j]:
            plat += 1
            i += 1
        else:
            plat -= 1
            j += 1
        best = max(best, plat)
    return best

print(min_platforms([900, 940, 950, 1100, 1500, 1800], [910, 1200, 1120, 1130, 1900, 2000]))   # 3

3


## 4. Car Fleet

Cars on a one-lane road heading to a `target`. A faster car behind a slower one **joins its fleet** and moves at the slower speed. Count the fleets.

Sort by position **closest to the target first**. Compute each car's **time to reach the target** alone. If a car behind would arrive **sooner or at the same time** than the fleet ahead, it merges into that fleet. Only a **strictly larger** time starts a new fleet.

In [4]:
def car_fleet(target, position, speed):
    cars = sorted(zip(position, speed), reverse=True)
    fleets, slowest = 0, 0
    for p, s in cars:
        t = (target - p) / s
        if t > slowest:                      # cannot catch the fleet ahead
            fleets += 1
            slowest = t
    return fleets

print(car_fleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]))   # 3

3


## 5. My Calendar I (design)

`book(start, end)` returns `True` and books if it does not double-book. Check against every existing booking with the overlap test. O(n) per call with a list; a sorted structure (BST / `bisect`) gets O(log n) search.

In [5]:
class MyCalendar:
    def __init__(self):
        self.booked = []

    def book(self, start, end):
        if any(start < e and s < end for s, e in self.booked):
            return False
        self.booked.append((start, end))
        return True

cal = MyCalendar()
print(cal.book(10, 20), cal.book(15, 25), cal.book(20, 30))   # True False True

True False True


## 6. Interview thinking

**Signals:** "maximum number of overlapping", "minimum rooms/platforms", "can all meetings be scheduled", "events in time".

**Say:** "I convert intervals into start/end events and sweep through time, tracking a counter. O(n log n) for sorting."

**Common mistakes:** end/start order at equal times, using `<=` vs `<` for touching intervals, forgetting to sort the events, forgetting that car-fleet times must be compared as floats.

## Quick Revision

- Activity selection: sort by end, take if `start >= last_end`.
- Sweep line: `+1` at start, `-1` at end, sort, running max.
- Platforms: sorted arrivals and departures, two pointers.
- Car fleet: sort by closeness to the target, compare times.